In [ ]:
!pip install torchtune
!pip install torchao

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchtune.modules import RotaryPositionalEmbeddings

**Flash Attention**

*   every third layer: FLashAttention + Global Attention
*   all other layers: FlashAttention + Local Attention





In [31]:
class FlashAttention(nn.Module):
  def __init__(self, d_model, d_k, d_v, num_head, block_size=128, window_size=128, use_local_attenion = False):
    super().__init__()
    self.d_k = d_k
    self.d_v = d_v
    self.num_head = num_head
    self.block_size = block_size
    self.window_size = window_size
    self.use_local_attenion = use_local_attenion


    self.W_q = nn.Linear(d_model, num_head * d_k, bias=False)
    self.W_k = nn.Linear(d_model, num_head * d_k, bias=False)
    self.W_v = nn.Linear(d_model, num_head * d_v, bias=False)
    self.W_o = nn.Linear(num_head * d_v, d_model, bias=False)

    self.rope = RotaryPositionalEmbeddings(dim=self.d_k, max_seq_len=512)


  def flash_attention(self, Q, K, V):
    """ Compute the flash attention between Q, K, V
      Inputs:
        Q (N, num_head, seq_length, d_k)
        K (N, num_head, seq_length, d_k)
        V (N, num_head, seq_length, d_v)

      Output:
        attention_output: (N, num_head, seq_length, d_v)
    """
    N, num_head, seq_length, d_k = Q.shape
    block_size = min(self.block_size, seq_length)

    attention_output = torch.zeros_like(V)

    if self.use_local_attenion == False:
      # flash attention with global attention
      for i in range(0, seq_length, block_size):
        for j in range(0, seq_length, block_size):
          Q_block = Q[:, :, i:i+block_size, :] # (N, num_head, block_size, d_k)
          K_block = K[:, :, i:i+block_size, :] # (N, num_head, block_size, d_k)
          V_block = V[:, :, i:i+block_size, :] # (N, num_head, block_size, d_v)

          QK_T = torch.matmul(Q_block, K_block.transpose(-2, -1)) / np.sqrt(self.d_k)  # (N, num_head, block_size, block_size)

          # numerical stability softmax mentioned in the paper
          # computing softmax(QK.T)
          QK_T = QK_T - torch.max(QK_T, dim=-1, keepdim=True)[0]
          attn_block = torch.exp(QK_T)
          attn_block = attn_block / torch.sum(attn_block, dim=-1, keepdim=True)  # (N, num_head, block_size, block_size)

          # appply attention to V block
          attn_V = torch.matmul(attn_block, V_block) # (N, num_head, block_size, d_v)


          attention_output[:, :, i:i+block_size, :] += attn_V

    else:
      # flash attention with local attention
      window = self.local_window_size
      # outerloop: get the current local window
      for win_start in range(0, seq_length, window):
        win_end = min(win_start + window, seq_length)
        Q_window = Q[:, :, win_start:win_end, :]  # (N, num_head, window, d_k)
        K_window = K[:, :, win_start:win_end, :]  # (N, num_head, window, d_k)
        V_window = V[:, :, win_start:win_end, :]  # (N, num_head, window, d_v)

        # divide the local window into Flash Attention blocks
        for qi in range(0, (win_end - win_start), block_size):
          qi_end = min(qi + block_size, win_end - win_start)
          Q_block = Q_window[:, :, qi:qi_end, :]
          for kj in range(0, (win_end - win_start), block_size):
            kj_end = min(kj + block_size, win_end - win_start)
            K_block = K_window[:, :, kj:kj_end, :]
            V_block = V_window[:, :, kj:kj_end, :]

            QK_T = torch.matmul(Q_block, K_block.transpose(-2, -1))
            QK_T = QK_T - torch.max(QK_T, dim=-1, keepdim=True)[0]
            attn_block = torch.exp(QK_T)
            attn_block = attn_block / torch.sum(attn_block, dim=-1, keepdim=True)

            attn_V = torch.matmul(attn_block, V_block)
            global_qi_start = win_start + qi
            global_qi_end   = win_start + qi_end


            attention_output[:, :, global_qi_start:global_qi_end, :] += attn_V

    return attention_output

  def forward(self, input):
    '''
    Inputs:
      input = Q, K, V: (N, seq_length, d_model)
    Output:
      attention (N, seq_length, d_model)
    '''
    # do the rest of the steps
    Q = K = V = input
    N, seq_length, _ = Q.shape
    Q = self.W_q(Q).view(N, seq_length, self.num_head, self.d_k).transpose(1, 2)  # (N, num_head, seq_length, d_k)
    K = self.W_k(K).view(N, seq_length, self.num_head, self.d_k).transpose(1, 2)  # (N, num_head, seq_length, d_k)
    V = self.W_v(V).view(N, seq_length, self.num_head, self.d_v).transpose(1, 2)  # (N, num_head, seq_length, d_v)

    # apply RoPE
    Q = self.rope.forward(Q)
    K = self.rope.forward(K)


    attention = self.flash_attention(Q, K, V) # (N, num_head, seq_length, d_v)

    attention = attention.transpose(1, 2).contiguous().view(N, seq_length, -1) # (N, seq_length, num_head * d_v)
    attention = self.W_o(attention)  # (N, seq_length, d_model)
    return attention

In [43]:
class AttentionLayer(nn.Module):
  def __init__(self, batch_size, seq_length, num_head, d_k, d_v, embedding_dim, use_local_attenion):
    super().__init__()
    self.layerNorm = nn.LayerNorm(embedding_dim)
    self.ff = nn.ModuleList([nn.Linear(embedding_dim, 2*embedding_dim), nn.Linear(2*embedding_dim, embedding_dim)])
    self.mha = FlashAttention(embedding_dim, d_k, d_v, num_head, block_size=128, window_size=128, use_local_attenion = False)

  def forward(self, x):
    ''' AttentionLayer of ModernBERT
    - use pre normalization
    - layer_num % 3 == 0: flashAttention3 + global attention
    - lyaer_num % 3 != 0: flashAttention2 + local attention(128)
    - no activation except decoder
    - use GeGLU activation
    - RoPE

    arguments:
      x: Q=K=V (N, seq_length, embedding_dim)
    output:
      x: (N, seq_length, embedding_dim)
    '''
    # x = Q, K, V
    x_prime = self.layerNorm(x)
    x = self.mha(x_prime)
    x = x + x_prime

    x_prime = self.layerNorm(x)
    x = x_prime
    for layer in self.ff:
      x = layer(x)

    x = x + x_prime

    return x

In ModernBERT bais term is disabled except the final layer and the finaly layer uses the GeGLU. ere final layer refers to `(N, seq_lemgth, embdding_dim) -> (N, seq_length, vocab_size)`

In [44]:
class GeGLU(nn.Module):
  def __init__(self, input_dim, output_dim):
    super(GeGLU, self).__init__()
    self.proj_gate = nn.Linear(input_dim, output_dim)
    self.proj_linear = nn.Linear(input_dim, output_dim)

  def forward(self, x):
    return F.gelu(self.proj_gate(x)) * self.proj_linear(x)

In [45]:
class ModernBERT(nn.Module):
  def __init__(self, num_layers, batch_size, seq_length, d_model, num_head, d_k, d_v, vocab_size):
    super().__init__()
    self.num_layers = num_layers
    self.input_embedding = nn.Embedding(vocab_size, d_model)
    self.attention_layers = nn.ModuleList([AttentionLayer(batch_size, seq_length, num_head, d_k, d_v, d_model, i % 3 == 0) for i in range(num_layers)])
    self.decoder_layer = GeGLU(d_model, vocab_size) # (N, seq-length, embedding_dim) -> (N, seq_length, vocab_size)


  def forward(self, x):

    # input embdding
    x = self.input_embedding(x) # (N, seq_length) -> (N, seq_length, d_model)
    # attention layers
    for layer in self.attention_layers:
      x = layer(x)

    # decoder layers
    x = self.decoder_layer(x)

    return x

In [47]:
vocab_size = 1000
d_model = 768
num_layers = 2
num_head = 12
d_k = d_v = d_model // num_head
batch_size = 1
seq_length = 32

model = ModernBERT(num_layers, batch_size, seq_length, d_model, num_head, d_k, d_v, vocab_size)

sample_input = torch.randint(0, vocab_size, (batch_size, seq_length))
output = model(sample_input)
print("Output shape:", output.shape)

Output shape: torch.Size([1, 32, 1000])
